In [22]:
import pandas as pd
from tokenizers import Tokenizer
pd.set_option("display.max_colwidth", None)

In [10]:
tokenizer = Tokenizer.from_file("/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json")
tokenizer.get_vocab_size()

45000

In [12]:
csv_path = "/kaggle/input/datasets/punitkashyap2007/virgo-general-csv/virgo_chat_dataset.csv"
chat_df = pd.read_csv(csv_path)

In [13]:
def format_chat(prompt, response):
    return (
        "<bos>\n"
        "User:\n"
        f"{str(prompt).strip()}\n\n"
        "Virgo:\n"
        f"{str(response).strip()}\n\n"
        "<eos>"
    )

chat_df["text"] = chat_df.apply(
    lambda row: format_chat(row["prompt"], row["response"]),
    axis=1
)

chat_df = chat_df[["text"]]

print(chat_df.iloc[0]["text"])

<bos>
User:
Can you tell me more about Forever Glass and how it's providing employment opportunities for autistic adults?

Virgo:
Forever Glass is a glass blowing business owned by Cathy Porter and her sister Bernadette Guimarin. The business is located in Placerville, Northern California, and it provides employment opportunities for developmentally disabled adults. Some of the employees at Forever Glass are autistic adults who have difficulty finding employment elsewhere due to their symptoms such as poor verbal skills, crowd anxiety, and trouble understanding instructions. Forever Glass offers a personalized "keepsake bowl" service, where customers can send recycled bottles from a wedding or other special event and have the company turn them into one-of-a-kind glass creations. As the business grows, the owners hope to hire a shop manager and step away from day-to-day glass production, envisioning a team of adults on the spectrum bustling around the family’s woods and gardens, each ho

In [14]:
chat_df.head()

,text
0,<bos>\nUser:\nCan you tell me more about Forev...
1,<bos>\nUser:\nwrite an original short story of...
2,<bos>\nUser:\nCan you tell me more about the c...
3,<bos>\nUser:\nCreate a mixed media artwork fea...
4,<bos>\nUser:\nThe sum of three distinct digits...


In [15]:
chat_df.shape

(1585857, 1)

In [16]:
chat_df = chat_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [18]:
from tqdm.auto import tqdm

all_tokens = []

for text in tqdm(chat_df["text"], desc="Tokenizing Chats"):
    token_ids = tokenizer.encode(text).ids
    all_tokens.extend(token_ids)

print(f"\n✅ Total Tokens: {len(all_tokens):,}")

Tokenizing Chats:   0%|          | 0/1585857 [00:00<?, ?it/s]


✅ Total Tokens: 591,200,102


In [19]:
import numpy as np

all_tokens = np.array(all_tokens, dtype=np.uint16)

print("Total Tokens:", len(all_tokens))
print("Memory:", all_tokens.nbytes / 1024**2, "MB")

Total Tokens: 591200102
Memory: 1127.6247062683105 MB


In [20]:
split_idx = int(len(all_tokens) * 0.995)

train_tokens = all_tokens[:split_idx]
val_tokens = all_tokens[split_idx:]

print(f"Train Tokens: {len(train_tokens):,}")
print(f"Val Tokens:   {len(val_tokens):,}")

Train Tokens: 588,244,101
Val Tokens:   2,956,001


In [21]:
train_tokens.tofile("chat_train.bin")
val_tokens.tofile("chat_val.bin")

print("✅ Saved chat_train.bin")
print("✅ Saved chat_val.bin")

✅ Saved chat_train.bin
✅ Saved chat_val.bin


In [27]:
import zipfile
import os

files_to_zip = [
    "/kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json",
    "chat_train.bin",
    "chat_val.bin"
]

zip_name = "virgo_chat_dataset.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file, arcname=os.path.basename(file))
            print(f"✅ Added: {file}")
        else:
            print(f"❌ Missing: {file}")

print(f"\n🎉 ZIP created successfully: {zip_name}")

✅ Added: /kaggle/input/datasets/punitkashyap2007/virgo-tokenizer/virgo_tokenizer.json
✅ Added: chat_train.bin
✅ Added: chat_val.bin

🎉 ZIP created successfully: virgo_chat_dataset.zip


In [28]:
from tqdm.auto import tqdm
import numpy as np

token_lengths = []

for text in tqdm(chat_df["text"], desc="Analyzing Token Lengths"):
    token_lengths.append(len(tokenizer.encode(text).ids))

token_lengths = np.array(token_lengths)

print("=" * 60)
print("           VIRGO CHAT DATASET STATISTICS")
print("=" * 60)

print(f"Total Conversations : {len(token_lengths):,}")
print(f"Total Tokens        : {token_lengths.sum():,}")
print(f"Average Tokens      : {token_lengths.mean():.2f}")
print(f"Median Tokens       : {np.median(token_lengths):.2f}")
print(f"Minimum Tokens      : {token_lengths.min():,}")
print(f"Maximum Tokens      : {token_lengths.max():,}")
print(f"Std Deviation       : {token_lengths.std():.2f}")

print("\nPercentiles")
print("-" * 60)
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"{p:>2}% : {np.percentile(token_lengths, p):8.2f} tokens")

print("\nLength Distribution")
print("-" * 60)

bins = [
    (0,64),
    (65,128),
    (129,256),
    (257,512),
    (513,1024),
    (1025,2048),
    (2049,4096),
    (4097,8192),
    (8193,float("inf"))
]

for low, high in bins:
    if high == float("inf"):
        count = np.sum(token_lengths >= low)
        label = f"{low}+"
    else:
        count = np.sum((token_lengths >= low) & (token_lengths <= high))
        label = f"{low}-{high}"

    print(f"{label:>12}: {count:>8,} ({count/len(token_lengths)*100:6.2f}%)")

print("\nLongest Conversation")
print("-" * 60)
idx = np.argmax(token_lengths)
print(f"Row Index : {idx}")
print(f"Tokens    : {token_lengths[idx]:,}")

print("\nShortest Conversation")
print("-" * 60)
idx = np.argmin(token_lengths)
print(f"Row Index : {idx}")
print(f"Tokens    : {token_lengths[idx]:,}")

Analyzing Token Lengths:   0%|          | 0/1585857 [00:00<?, ?it/s]

           VIRGO CHAT DATASET STATISTICS
Total Conversations : 1,585,857
Total Tokens        : 591,200,102
Average Tokens      : 372.80
Median Tokens       : 318.00
Minimum Tokens      : 25
Maximum Tokens      : 7,987
Std Deviation       : 264.29

Percentiles
------------------------------------------------------------
 1% :    51.00 tokens
 5% :    86.00 tokens
10% :   119.00 tokens
25% :   200.00 tokens
50% :   318.00 tokens
75% :   472.00 tokens
90% :   673.00 tokens
95% :   853.00 tokens
99% :  1263.00 tokens

Length Distribution
------------------------------------------------------------
        0-64:   36,815 (  2.32%)
      65-128:  148,733 (  9.38%)
     129-256:  388,336 ( 24.49%)
     257-512:  683,011 ( 43.07%)
    513-1024:  285,317 ( 17.99%)
   1025-2048:   40,591 (  2.56%)
   2049-4096:    3,020 (  0.19%)
   4097-8192:       34 (  0.00%)
       8193+:        0 (  0.00%)

Longest Conversation
------------------------------------------------------------
Row Index : 1241691